# Digit Recognizer — GPU baseline

로컬 Kaggle GPU Dev Container와 Kaggle 원격 Jupyter 커널에서 같은 노트북을 실행합니다.

- 로컬: `kagglehub`로 대회 데이터를 `digit-recognizer/data/`에 다운로드합니다. 이 폴더는 Ubuntu 저장소에 유지됩니다. Kaggle 인증 및 대회 규칙 동의가 필요할 수 있습니다.
- Kaggle: Digit Recognizer 대회 데이터를 연결하고 GPU accelerator를 활성화합니다.
- 첫 셀의 실행 환경과 GPU 이름으로 **선택한 커널**을 확인한 뒤 학습합니다.
- 결과는 프로젝트의 `output/` 폴더에 저장합니다. 로컬 실행에서는 Ubuntu 저장소에 유지됩니다. 원격 실행 결과는 서버에 저장됩니다.
- 패키지는 이미지에 포함된 것을 사용합니다. HTML 내보내기는 로컬·원격 실행 검증 이후에 진행합니다.


In [ ]:
import os
import sys
import time
import random
from pathlib import Path
from importlib.metadata import version

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay

# fix seed to reproduce results

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)

# resource check
print("Host:", os.uname().nodename)
print("Python:", sys.version)
print("Interpreter:", sys.executable)

assert torch.cuda.is_available()
device = torch.device("cuda")
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.version.cuda)

print(
    {p: version(p) for p in ["torch", "numpy", "pandas", "scikit-learn", "matplotlib"]}
)

probe = torch.arange(16, dtype=torch.float32, device=device)
assert (probe @ probe).item() == 1240.0
print("CUDA computation passed")


## 데이터 준비

프로젝트의 `data/` 폴더에 `train.csv`와 `test.csv`가 있으면 그대로 사용합니다.
로컬 다운로드 인증은 저장소 루트의 `.env`에 `KAGGLE_API_TOKEN=토큰` 형식으로 설정합니다.
`.env.example`을 `.env`로 복사한 뒤 값을 입력하세요. `.env`는 Git에서 제외됩니다.
`load_dotenv()`로 `.env`를 환경 변수에 로드하고 `os.environ`으로 토큰을 읽습니다.
이미 설정된 환경 변수는 `.env`보다 우선하며, 토큰은 출력하지 않습니다.
`.env`의 인증 값을 바꾼 뒤에는 커널을 재시작하고 다시 실행하세요. 인증 실패 시 토큰과 대회 규칙 동의를 확인하세요.
로컬 데이터는 마운트된 Ubuntu 저장소에 유지됩니다.


In [ ]:
import kagglehub
from dotenv import load_dotenv
from kagglehub.auth import set_kaggle_api_token
from kagglehub.exceptions import UnauthenticatedError

project_dir = (
    Path.cwd()
    if Path.cwd().name == "digit-recognizer"
    else Path.cwd() / "digit-recognizer"
)
data_dir = project_dir / "data"

if not all((data_dir / name).is_file() for name in ("train.csv", "test.csv")):
    env_file = project_dir.parent / ".env"
    load_dotenv(env_file, override=False)
    token = os.environ.get("KAGGLE_API_TOKEN", "").strip()
    if token:
        set_kaggle_api_token(token)
    del token

    data_dir.mkdir(parents=True, exist_ok=True)

    try:
        kagglehub.competition_download(
            "digit-recognizer", output_dir=str(data_dir.resolve())
        )
    except UnauthenticatedError:
        raise RuntimeError(
            "Kaggle 인증 실패: 저장소 루트 .env의 KAGGLE_API_TOKEN과 대회 규칙 동의를 확인한 뒤 이 셀을 다시 실행하세요."
        ) from None

print("Competition data:", data_dir.resolve())


In [ ]:
output_dir = project_dir / "output"
output_dir.mkdir(parents=True, exist_ok=True)
train = pd.read_csv(data_dir / "train.csv")
test = pd.read_csv(data_dir / "test.csv")

assert "label" in train.columns
pixels = [f"pixel{i}" for i in range(784)]

assert list(train.drop(columns="label").columns) == pixels
assert list(test.columns) == pixels
assert train["label"].between(0, 9).all()
assert not train.isna().any().any() and not test.isna().any().any()
assert train[pixels].min().min() >= 0 and train[pixels].max().max() <= 255
assert test.min().min() >= 0 and test.max().max() <= 255

print("Data:", data_dir.resolve())
print("Shapes:", train.shape, test.shape)
display(train.head())


In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4))

for ax, i in zip(axes.flat, range(10)):
    ax.imshow(
        train.loc[i, pixels].to_numpy(dtype=np.float32).reshape(28, 28), cmap="gray"
    )
    ax.set_title(f"Label: {train.loc[i, 'label']}")
    ax.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
X = train[pixels].to_numpy(dtype=np.float32) / 255.0
y = train["label"].to_numpy(dtype=np.int64)
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)


def loader(x, y, shuffle=False):
    return DataLoader(
        TensorDataset(torch.from_numpy(x).reshape(-1, 1, 28, 28), torch.from_numpy(y)),
        batch_size=256,
        shuffle=shuffle,
        num_workers=0,
        generator=torch.Generator().manual_seed(SEED),
    )


train_loader = loader(X_train, y_train, True)
valid_loader = loader(X_valid, y_valid)

model = nn.Sequential(
    nn.Conv2d(1, 16, 3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Conv2d(16, 32, 3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Flatten(),
    nn.Linear(32 * 7 * 7, 128),
    nn.ReLU(),
    nn.Linear(128, 10),
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

EPOCHS = 3
history = []

for epoch in range(1, EPOCHS + 1):
    started = time.perf_counter()
    model.train()
    total_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(labels)

    model.eval()
    with torch.inference_mode():
        predictions = np.concatenate(
            [
                model(images.to(device)).argmax(1).cpu().numpy()
                for images, _ in valid_loader
            ]
        )

    row = dict(
        epoch=epoch,
        loss=total_loss / len(X_train),
        accuracy=accuracy_score(y_valid, predictions),
        seconds=time.perf_counter() - started,
    )
    history.append(row)
    print(row)

display(pd.DataFrame(history))


In [ ]:
ConfusionMatrixDisplay.from_predictions(y_valid, predictions)
plt.title("Validation confusion matrix")
plt.show()

assert len(predictions) == len(y_valid)
assert np.isfinite([r["loss"] for r in history]).all()

test_x = test.to_numpy(dtype=np.float32) / 255.0
model.eval()
with torch.inference_mode():
    test_predictions = np.concatenate(
        [
            model(
                torch.from_numpy(test_x[i : i + 256]).reshape(-1, 1, 28, 28).to(device)
            )
            .argmax(1)
            .cpu()
            .numpy()
            for i in range(0, len(test_x), 256)
        ]
    )
submission = pd.DataFrame(
    {"ImageId": np.arange(1, len(test_predictions) + 1), "Label": test_predictions}
)
assert len(submission) == len(test) and submission["Label"].between(0, 9).all()

submission.to_csv(output_dir / "submission.csv", index=False)
pd.DataFrame(history).to_csv(output_dir / "training_history.csv", index=False)

print("Saved:", output_dir / "submission.csv")
display(submission.head())
